# Create a route and display HOSTRADA/CERRA climate variables
This notebook covers the complete workflow:

1. Select weather provider and route domain
2. Create the route (origin, destination, travel profile, and departure time). Calculate the intermediate route points and save them directly as a CSV file.
4. Calculate one or more HOSTRADA or CERRA climate variables along the route.
5. Display the selected variable as a time series, summary table, and map.

OSRM is used for driving, cycling, and walking. For rail/public transport, OpenTripPlanner is tried first; if it is unavailable, the existing transit fallback is used.

In [ ]:
from pathlib import Path
import folium
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from ipywidgets import Dropdown, SelectMultiple, VBox, Output, HTML
from hostrada4py.hostradaRoute import HOSTRADA_VARIABLES, available_variables, calculate_route_climate

## 1. Select weather provider and route domain

Choose the weather-data provider **before** defining the route. DWD/HOSTRADA supports routes inside Germany; CERRA supports routes across Europe.


In [ ]:
from hostrada4py import providerUI as provider_ui

provider_selector = provider_ui.create_provider_selector(
    globals(),
    initial="dwd",
    title="1. Select weather-data source",
    show=True,
)

route_domain_hint = HTML()
display(route_domain_hint)

def _sync_route_provider(change=None):
    global route_provider
    route_provider = str(provider_dropdown.value)
    if route_provider == "dwd":
        route_domain_hint.value = (
            "<b>Route domain:</b> Germany. Start and destination must both be in Germany. "
            "For European routes, select CERRA before creating the route."
        )
    else:
        route_domain_hint.value = (
            "<b>Route domain:</b> Europe. Start and destination may be selected in European countries."
        )

    app = globals().get("route_app")
    if app is not None and hasattr(app, "set_provider"):
        app.set_provider(route_provider)

    selector = globals().get("variable_selector")
    if selector is not None and hasattr(selector, "options"):
        supported = set(available_variables(provider=route_provider)["variable"])
        filtered = [
            (HOSTRADA_VARIABLES[code]["label"], code)
            for code in HOSTRADA_VARIABLES
            if code in supported
        ]
        old_values = tuple(value for value in selector.value if value in supported)
        selector.value = ()
        selector.options = filtered
        available_codes = [value for _, value in filtered]
        selector.value = old_values or (("tas",) if "tas" in available_codes else tuple(available_codes[:1]))

provider_dropdown.observe(_sync_route_provider, names="value")
_sync_route_provider()


## 2. Create the route and calculate the intermediate points

Run the next cell only after selecting the provider above. The route form uses the corresponding Germany or Europe address-search domain.


In [ ]:
from hostrada4py.routeLeafletApp import (
    ROUTE_APP_PROVIDER_REVISION,
    RouteAppDefaults,
    launch_route_app,
)

route_provider = str(provider_dropdown.value)
if route_provider == "dwd":
    default_start = "Einsteinufer 43-53, 10587 Berlin, Deutschland"
    default_destination = "München, Deutschland"
else:
    default_start = "Berlin, Deutschland"
    default_destination = "Paris, Frankreich"

route_defaults = RouteAppDefaults(
    start_address=default_start,
    destination_address=default_destination,
    start_time="2025-08-01T12:00+00:00",
    average_speed_kmh=100.0,
    interval_minutes=30,
    profile="driving",
    language="de",
    output_file="route_positions.csv",
    provider=route_provider,
    otp_url="http://localhost:8080/otp/gtfs/v1",
)
route_app = launch_route_app(route_defaults)


### Use the CSV file from the route form

In [ ]:
route_csv = Path(route_app.output_file.value or "route_positions.csv")
output_csv = route_csv.with_name(f"{route_csv.stem}_climate.csv")
if not route_csv.exists():
    raise FileNotFoundError("The route CSV has not been created yet. Click 'Calculate route' above and run this cell again. " + f"Expected file: {route_csv.resolve()}")
print(f"Route CSV: {route_csv.resolve()}")
print(f"Climate output: {output_csv.resolve()}")

## 3. Select and calculate climate variables


## Available climate variables

In [ ]:
display(available_variables(provider=route_provider))


## Select climate variables for calculation

In [ ]:
supported_codes = set(available_variables(provider=route_provider)["variable"])
variable_selector = SelectMultiple(
    options=[
        (meta["label"], code)
        for code, meta in HOSTRADA_VARIABLES.items()
        if code in supported_codes
    ],
    value=("tas",), description="Variables:", rows=11,
)
display(variable_selector)

In [ ]:
selected_variables = tuple(variable_selector.value)
if not selected_variables:
    raise ValueError("Select at least one climate variable.")
def show_progress(step, total, _row, variable):
    print(f"\rCalculation {step}/{total}: {variable}", end="", flush=True)
climate_data = calculate_route_climate(
    route_csv, variables=selected_variables, output_csv=output_csv,
    # One shared DWD monthly file is reused for every route point.
    cache_strategy="full", continue_on_error=True,
    progress_callback=show_progress, provider=route_provider,
)
print(f"\nSaved: {output_csv.resolve()}")
climate_data["timestamp"] = pd.to_datetime(climate_data["timestamp"], errors="coerce")
display(climate_data.head(5))

## Select a climate variable for display

In [ ]:
plot_selector = Dropdown(
    options=[(HOSTRADA_VARIABLES[code]["label"], code) for code in selected_variables],
    value=selected_variables[0], description="Chart:",
)
display(plot_selector)

In [ ]:
variable = plot_selector.value
metadata = HOSTRADA_VARIABLES[variable]
value_column = metadata["output_column"]
plot_data = climate_data.dropna(subset=["timestamp", value_column]).sort_values("timestamp")
route_start = plot_data["timestamp"].iloc[0]
plot_data = plot_data.copy()
plot_data["elapsed_minutes"] = (plot_data["timestamp"] - route_start).dt.total_seconds()/60
fig, ax = plt.subplots(figsize=(12,5))
ax.plot(plot_data["elapsed_minutes"], plot_data[value_column], marker="o")
ax.set_title(f"{metadata['label']} during the trip")
ax.set_xlabel("Elapsed travel time [min]")
ax.set_ylabel(f"{metadata['short_label']} [{metadata['display_unit']}]")
ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

summary = pd.DataFrame({
    "Statistic": ["Initial value","Final value","Minimum","Maximum","Mean","Change from start → destination"],
    f"Value [{metadata['display_unit']}]": [
        plot_data[value_column].iloc[0], plot_data[value_column].iloc[-1],
        plot_data[value_column].min(), plot_data[value_column].max(),
        plot_data[value_column].mean(), plot_data[value_column].iloc[-1]-plot_data[value_column].iloc[0],
    ],
})
display(summary.round(3))

## Map of the selected climate variable

In [ ]:
map_data = plot_data.dropna(subset=["latitude", "longitude", value_column])
route_map = folium.Map(location=[map_data["latitude"].iloc[0], map_data["longitude"].iloc[0]], zoom_start=8, control_scale=True)
folium.PolyLine(map_data[["latitude","longitude"]].values.tolist(), weight=4, opacity=0.8, tooltip="Route").add_to(route_map)
for _, row in map_data.iterrows():
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]], radius=5, fill=True, fill_opacity=0.8,
        tooltip=f"{row['timestamp'].strftime('%Y-%m-%d %H:%M')} · {row[value_column]:.2f} {metadata['display_unit']}",
        popup=f"<b>{metadata['label']}</b><br>{row[value_column]:.3f} {metadata['display_unit']}<br><b>Time:</b> {row['timestamp']}",
    ).add_to(route_map)
route_map.fit_bounds(map_data[["latitude","longitude"]].values.tolist())
route_map